In [1]:
import torch
import polars as pl

In [2]:
# loading the dataset from hugging face
from datasets import load_dataset
dataset = load_dataset("mohamed-khalil/ATHAR")

In [3]:
#separating the data into training and testing data
train_data=dataset['train'].to_polars()
test_data=dataset['test'].to_polars()

- Note: Stop words won't be removed as we want to test them in the translation

In [4]:
# manually clean the data from links and invalid values through regular expression
# add padding to indicate beginning and end of the words
# tokenize the words
# produce a list of array polars fields
def tokenize_native(t):
  # /s to match any whitespace
  t=t.with_columns(
    pl.all()
    .str.to_lowercase() # convert the english letters to lower case
    .str.replace_all(r'http\S+|www\S+|@|#', '') # strip out the links and special characters
    .str.replace_all(r'^\w\s', ' ') # delete any non alphanumeric word followed by a space
    .str.replace_all(r'\s+', ' ') # delete any consecutive spaces
  ).with_columns(
    pl.format('<sos> {} <eos>', pl.col('arabic')), # add beginning and ending of the sentence from right to left
    pl.format('<sos {} <eos>', pl.col('english')) # add beginning and ending of the sentence from left to right
  ).with_columns(
      pl.all().str.split(' ') # tokenize words
  )
  return t

train_data=tokenize_native(train_data)
test_data=tokenize_native(test_data)

In [5]:
# see what the sentnece is like in both arabic and english
def sen(d, index):
    if isinstance(d['arabic'][index], pl.Series):
        arabic=d['arabic'][index].to_list()
        english=d['english'][index].to_list()
    else:
        arabic=d['arabic'][index].split(' ')
        english=d['english'][index].split(' ')
    for ar, eng in zip(arabic, english):
        print(f"Arabic: {ar}, English: {eng}")

In [6]:
# Pad polars list to the maximum one
# find the maximum list
# Add in text words to ensure all list poems are within the same length
def padding_df(t):
# add padding to the data to ensure they are compatible to be converted to a tensor type
  for col in t.columns:
    max=t.select(pl.col(col)).with_columns(pl.col(col).list.len()).max().item(0,0) # take the maximum length of the list
    t=t.with_columns(
        pl.col(col).list.concat(
            pl.lit("<pad>").repeat_by(max-pl.col(col).list.len()) # to ensure compatibleness, repeat by whole size - list len
            # for example if the list len is 3 max is 8 then add only five elements
        )
    )
  return t

train_data=padding_df(train_data)
test_data=padding_df(test_data)


In [7]:
# polars_native way to encode the values --not efficient at all
def encode(d):
    return d.with_columns(
        pl.all().cast(pl.List(pl.Categorical)).to_physical()
    )

In [8]:
# cast the values to an integer for easier tensor conversion
def cast_Utf(t):
  for col in t.columns:
    max=t.select(pl.col(col)
    ).with_columns(
        pl.col(col).list.len()
        ).max().item(0,0) # take the maximum length of the list
    t=t.with_columns(
      pl.col(col).cast(pl.Array(pl.UInt32, shape=(max)))
    )

  return t

In [9]:
# train_data=cast_Utf(encode(train_data))
# test_data=cast_Utf(encode(test_data))

In [10]:
# import torch
# from sklearn.model_selection import train_test_split
# x_train, x_val, y_train, y_val= train_test_split(train_data['arabic'], train_data['english'], test_size=0.3)
# x_train, x_val, y_train, y_val=x_train.to_torch().to(torch.long), x_val.to_torch().to(torch.long), y_train.to_torch().to(torch.long), y_val.to_torch().to(torch.long)

In [11]:
# #TODO: prepare the training engine
# #TODO: prepare the inference engine
# # code for inference to get back the data to the original format
# train_data.with_columns(pl.all().cast(pl.List(pl.Categorical)))

In [12]:
import torch
from torch.utils.data import Dataset
# other way to encode the data --efficient
# words index history is saved
# This method is efficient for both inference and training
# the most complete structure to store words and their frequency
class nuc_dataset(Dataset):
    def __init__(self,d):
        self.polars_series=d
        self.word2index={'<sos>': 0, '<eos>':1, '<pad>':2}
        self.index2word={0: '<sos>', 1: '<eos>', 2: '<pad>'}
        self.word2count={'<sos>': 0, '<eos>':0, '<pad>':0}
        self.counter=3 # counts of the existing words
        self.sentences={}
        self.tensor=[[]]
    
    # get the total length of the data
    def __len__(self):
        return self.polars_series.shape[0]

    def fill_words(self, sens):
        for word in sens:
            if word not in self.word2index:
                self.word2index[word]=self.counter
                self.index2word[self.counter]=word
                self.word2count[word]=1
                self.counter+=1
            else:
                self.word2count[word]+=1
                
    def encode_words(self):
        for i in range(self.__len__()):
            self.sentences[i]=self.polars_series.item(i).to_list()
            self.fill_words(self.sentences[i])
        return self

    def return_tensor(self):
        self.tensor=[
            [
            self.word2index[word] for word in self.sentences[i]
            ]
            for i in range(self.__len__())
        ]
        self.tensor=torch.tensor(self.tensor, dtype=torch.long)
        return self

    def __getitem__(self, idx):
        return self.tensor[idx]
        

In [13]:
def prepare_datasets(d):
    arabic_data=nuc_dataset(d['arabic']).encode_words()
    english_data=nuc_dataset(d['english']).encode_words()
    return arabic_data, english_data

ar_train, en_train=prepare_datasets(train_data)
ar_test, en_test=prepare_datasets(test_data)

In [14]:
ar_train=ar_train.return_tensor()
en_train=en_train.return_tensor()
ar_test=ar_test.return_tensor()
en_test=en_test.return_tensor()

In [15]:
assert ar_train.word2index['<pad>']==en_train.word2index['<pad>'], "Results are not the same"
pad_index=ar_train.word2index['<pad>']

## Data Analysis and Preprocessing
- in this part you are required to to conduct proper analysis of the above data
- You are also required to preprocess the above data in manner where is ready for modelling

## Modelling Section
- In this part you are required to build two models transformer and Attention based sequence to sequence model.

In [16]:
# # make all data parameteric on the type of device available either cuda or cpu
# device='cuda' if torch.cuda.is_available() else 'cpu'
# x_train=x_train.to(device)
# y_train=y_train.to(device)
# x_val=x_val.to(device)
# y_val=y_val.to(device)

## Attention Based Sequence to Sequence Model

In [17]:
import torch.nn as nn

class Encoder(nn.Module):
  def __init__(self, input_size, embedding_size, hidden_size, num_layers, p, device):
    super(Encoder, self).__init__()
    self.hidden_size=hidden_size
    self.num_layers=num_layers
    self.dropout=nn.Dropout(p)
    self.embedding=nn.Embedding(num_embeddings=input_size, embedding_dim=embedding_size, device=device)
    self.lstm=nn.LSTM(embedding_size, hidden_size, batch_first=True, num_layers=self.num_layers, device=device) # maybe you don't need to initialize a dropout at that layer

  def forward(self, x):
    # x is [batches, sentences]
    output=self.embedding(x)
    # x is now [batches, sentences, words_embeddings]
    output=self.dropout(output)
    # To prevent overfitting
    _, (hidden, cell)=self.lstm(output)
    # output is [batch, sentences, embeddings]
    # however, we don't want the output we the last hidden state and cell state
    # each of which represents the long and short term memories of the network
    # they help in buildling the context in the decoder.
    return hidden, cell

In [18]:
class Decoder(nn.Module):
  def __init__(self, input_size, embedding_size, hidden_size, num_layers, output_size, p, device):
    super(Decoder, self).__init__()
    self.output_size=output_size
    self.embedding_size=embedding_size
    self.num_layers=num_layers
    self.dropout=nn.Dropout(p)
    # note we are using input size to fed the embedding layer as it represents the index not the actual dimension
    self.embedding=nn.Embedding(num_embeddings=output_size, embedding_dim=embedding_size, device=device)
    # the embedding layer is responsible of building representative matrix for each word in the sequence
    # [batch, num_words, embedding_size]
    self.lstm=nn.LSTM(embedding_size, hidden_size, batch_first=True, num_layers=self.num_layers, device=device)
    # [batch, num_words, hidden_size]
    self.out=nn.Linear(hidden_size, output_size, device=device)
    # [batch, hidden_size, output_size]
  def forward(self, input, hidden, cell):
    # Inject the data the the embedding layer and then the dropout layer
      # the input shape given is [batch, sentence]
      embed=self.embedding(input)
      # batch is 1 because its predicting one sentence at a time
      # hidden and cell need refactoring because they are obtained for one sentence so
      embed=self.dropout(embed) # prevent overfitting
      output, (hidden, cell)=self.lstm(embed, (hidden, cell))
      # Use the same stored hidden and cell states to decode the data
      output=self.out(output)
      return output, hidden, cell

In [19]:
import random
from torch.nn.functional import log_softmax
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder=encoder
        self.decoder=decoder

    def forward(self, encoder_input, targets, teacher_forcing_ratio):
        # input shape is [batch, sequence_length, input]
        batch_length=encoder_input.shape[0] # take the total length
        sentence_length=targets.shape[1] # take the length of each sentence
        vocabs_length=self.decoder.output_size
        outputs=[]
        # outputs=torch.zeros(batch_length, sentence_length, vocabs_length)
        hidden, context= self.encoder(encoder_input)
        decoder_input=targets[:,0].long() # take the first word
        for i in range (1, sentence_length):
            # now we run the decoder
            decoder_output, hidden, context=self.decoder(decoder_input.unsqueeze(1), hidden, context) # I think I should change the encoder hidden and cell
            # each loop returns one word at a time
            # now we add this word to the outputs
            
            outputs.append(decoder_output)
            top_word=decoder_output.squeeze(1).argmax(1)
            teacher_force=random.uniform(0.0, 1.0) < teacher_forcing_ratio 
            # if the teacher_force_ration bigger than the generated random then we take the actual output as the next input, else we take the top generated word as the input
            decoder_input=targets[:, i] if teacher_force else top_word
        decoder_outputs=torch.cat(outputs, dim=1)
        decoder_outputs=log_softmax(decoder_outputs, dim=1)
        return decoder_outputs

In [20]:
# prepare the data
from torch.utils.data import DataLoader
def prepare_data(data) -> DataLoader:
    return DataLoader(data, batch_size=10, shuffle=False, num_workers=2)

In [21]:
import torch.optim as optim
def prepare_optim(params, lr, decay, pad_index):
    opt=optim.AdamW(params, lr=lr, weight_decay=decay)
    criterion=nn.NLLLoss(ignore_index=pad_index)
    return opt, criterion

In [22]:
# prepare the parameters
device='cuda' if torch.cuda.is_available else 'cpu'
input_size=ar_train.counter # set the input size based on the largest element for the embedding layer
embedding_size=128 # the dimensions of the embedding layer
hidden_size=256 # the output of the encoder
num_layers=1
dropout=0.5
output_size=en_train.counter
encoder=Encoder(input_size=input_size, embedding_size=embedding_size, hidden_size=hidden_size, num_layers=num_layers, p=dropout, device=device)
decoder=Decoder(input_size=input_size, embedding_size=embedding_size, hidden_size=hidden_size, num_layers=num_layers, output_size=output_size, p=dropout, device=device)

In [23]:

train_x = prepare_data(ar_train.tensor)
train_y = prepare_data(en_train.tensor)
val_x   = prepare_data(ar_test.tensor)
val_y   = prepare_data(en_test.tensor) 
model=Seq2Seq(encoder, decoder, device=device)

optim, loss=prepare_optim(model.parameters(), 0.001, 0.003, pad_index)

## The Method for training follows the Rust burn format

In [ ]:
num_epochs=15
for i in range(num_epochs):
    train_loss=0
    val_loss=0
    # training
    model.train()
    for x_t, y_t in zip(train_x, train_y):
        optim.zero_grad()
        x_t, y_t= x_t.to(device), y_t.to(device)
        output=model(x_t, y_t,0.2)
        grad=loss(output.view(-1, output.size(-1)), y_t[:, 1:].reshape(-1))
        grad.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optim.step()
        train_loss+=grad.item()
        print(f'print-- training so far: {train_loss}')

    # evaluation
    model.eval()
    with torch.no_grad():
       for x_v, y_v in zip(val_x, val_y):
            x_v, y_v= x_v.to(device), y_v.to(device)
            output=model(x_v, y_v, 0.0)
            grad=loss(output.view(-1, output.size(-1)),y_v[:, 1:].reshape(-1))
            val_loss+=grad.item()
            print(f'print-- validation so far: {val_loss}')
           
    if (len(train_x) == len(train_y)):
        train_loss=train_loss / len(train_x)
    if ( len(val_x) == len(val_y)):
        val_loss=val_loss / len(val_x)
    print(f"---Train Loss :{train_loss}---")
    print(f"---Validation Loss: {val_loss}---")

print-- training so far: 21176.77965271473
print-- training so far: 21179.623176693916
print-- training so far: 21182.619586586952
print-- training so far: 21185.356040358543
print-- training so far: 21188.044981360435
print-- training so far: 21190.936094403267
print-- training so far: 21194.241100907326
print-- training so far: 21197.61348259449
print-- training so far: 21200.703072190285
print-- training so far: 21203.85752427578
print-- training so far: 21206.659234404564
print-- training so far: 21209.497932076454
print-- training so far: 21212.32359468937
print-- training so far: 21215.289965987206
print-- training so far: 21217.993118166924
print-- training so far: 21220.93378174305
print-- training so far: 21223.538694500923
print-- training so far: 21226.833132386208
print-- training so far: 21229.8884421587
print-- training so far: 21232.652400374413
print-- training so far: 21235.524422049522
print-- training so far: 21238.603647351265
print-- training so far: 21241.59938132